# Speech Processing

# Assignment 7

# Name: Saran Jayakumar Palani
# Reg No: BL.EN.U4AIE23131

## Objective
1. Compute the real cepstrum of a speech signal from `test.wav`.
2. Plot and interpret the cepstrum of a selected frame.
3. Estimate pitch period and fundamental frequency from voiced frames using cepstral peak search.
4. Apply cepstral liftering to separate vocal tract and excitation contributions.
5. Draw detailed inferences from each stage of the experiment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile

plt.style.use('seaborn-v0_8-whitegrid')

## Objective 1: Load Speech Signal and Compute Real Cepstrum

In [ ]:
fs, x = wavfile.read('test.wav')
x = x.astype(np.float64)
if x.ndim > 1:
    x = np.mean(x, axis=1)
x = x / (np.max(np.abs(x)) + 1e-12)
t = np.arange(len(x)) / fs

frame_len = int(0.03 * fs)
hop = int(0.01 * fs)

def real_cepstrum(frame):
    frame = frame - np.mean(frame)
    frame = frame * np.hamming(len(frame))
    nfft = int(2 ** np.ceil(np.log2(len(frame) * 2)))
    X = np.fft.fft(frame, n=nfft)
    log_mag = np.log(np.abs(X) + 1e-10)
    c = np.real(np.fft.ifft(log_mag))
    return c, nfft

frames = []
energies = []
zcrs = []
for s in range(0, len(x) - frame_len, hop):
    fr = x[s:s + frame_len]
    frames.append(fr)
    energies.append(np.mean(fr ** 2))
    zcrs.append(np.mean(np.abs(np.diff(np.sign(fr)))) / 2)

frames = np.array(frames)
energies = np.array(energies)
zcrs = np.array(zcrs)

voiced_candidates = np.where((energies > np.percentile(energies, 60)) & (zcrs < np.percentile(zcrs, 45)))[0]
if len(voiced_candidates) == 0:
    voiced_idx = int(np.argmax(energies))
else:
    voiced_idx = int(voiced_candidates[len(voiced_candidates) // 2])

frame = frames[voiced_idx]
c, nfft = real_cepstrum(frame)
q = np.arange(nfft) / fs

print('Sampling rate (Hz):', fs)
print('Signal duration (s):', np.round(len(x) / fs, 3))
print('Frame length (samples):', frame_len)
print('Selected voiced frame index:', voiced_idx)

## Objective 2: Cepstrum Plot of Selected Frame

In [ ]:
q_ms = q * 1000
max_q_ms = 25
mask = q_ms <= max_q_ms

fig, ax = plt.subplots(2, 1, figsize=(12, 7))
time_frame = np.arange(len(frame)) / fs * 1000
ax[0].plot(time_frame, frame, color='tab:blue')
ax[0].set_title('Selected Speech Frame (Voiced)')
ax[0].set_xlabel('Time (ms)')
ax[0].set_ylabel('Amplitude')

ax[1].plot(q_ms[mask], c[mask], color='tab:red')
ax[1].set_title('Real Cepstrum of Selected Frame')
ax[1].set_xlabel('Quefrency (ms)')
ax[1].set_ylabel('Cepstral Amplitude')
ax[1].set_xlim(0, max_q_ms)

plt.tight_layout()
plt.show()

The low-quefrency region contains slowly varying vocal tract envelope information, while a distinct peak in the pitch quefrency range indicates periodic excitation due to vocal fold vibration.

## Objective 3: Pitch Period and Fundamental Frequency Estimation

In [ ]:
qmin = 0.005
qmax = 0.020
i0 = int(np.floor(qmin * fs))
i1 = int(np.ceil(qmax * fs))
search_region = c[i0:i1 + 1]
peak_local = int(np.argmax(search_region))
peak_idx = peak_local + i0
pitch_period = peak_idx / fs
f0 = 1.0 / pitch_period

plt.figure(figsize=(11, 4))
plt.plot(q_ms[mask], c[mask], color='tab:red', linewidth=1.5)
plt.axvline(pitch_period * 1000, color='tab:green', linestyle='--', linewidth=2, label='Cepstral peak')
plt.axvspan(qmin * 1000, qmax * 1000, color='tab:gray', alpha=0.2, label='Pitch search range')
plt.title('Cepstral Pitch Peak Detection (5 ms to 20 ms)')
plt.xlabel('Quefrency (ms)')
plt.ylabel('Cepstral Amplitude')
plt.xlim(0, 25)
plt.legend()
plt.show()

print('Estimated pitch period (s):', np.round(pitch_period, 6))
print('Estimated pitch period (ms):', np.round(pitch_period * 1000, 3))
print('Estimated fundamental frequency F0 (Hz):', np.round(f0, 2))

## Objective 4: Cepstral Liftering for Source-Filter Separation

In [ ]:
X = np.fft.fft(frame * np.hamming(len(frame)), n=nfft)
log_mag = np.log(np.abs(X) + 1e-10)

q_cut_ms = 3.0
q_cut = int(np.round((q_cut_ms / 1000) * fs))

low_c = np.zeros_like(c)
low_c[:q_cut + 1] = c[:q_cut + 1]
low_c[-q_cut:] = c[-q_cut:]
high_c = c - low_c

low_log_mag = np.real(np.fft.fft(low_c))
high_log_mag = np.real(np.fft.fft(high_c))
freq = np.fft.fftfreq(nfft, d=1 / fs)
pos = freq >= 0

fig, ax = plt.subplots(2, 1, figsize=(12, 8))
ax[0].plot(freq[pos], log_mag[pos], color='black', linewidth=1.2, label='Original log spectrum')
ax[0].plot(freq[pos], low_log_mag[pos], color='tab:blue', linewidth=2, label='Low-pass liftered (vocal tract)')
ax[0].set_title('Low-Pass Cepstral Liftering: Spectral Envelope')
ax[0].set_xlabel('Frequency (Hz)')
ax[0].set_ylabel('Log Magnitude')
ax[0].set_xlim(0, 5000)
ax[0].legend()

ax[1].plot(freq[pos], high_log_mag[pos], color='tab:orange', linewidth=1.6, label='High-pass liftered (excitation)')
ax[1].set_title('High-Pass Cepstral Liftering: Excitation Fine Structure')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('Log Magnitude')
ax[1].set_xlim(0, 5000)
ax[1].legend()

plt.tight_layout()
plt.show()

print('Lifter cutoff (ms):', q_cut_ms)
print('Lifter cutoff (samples in quefrency domain):', q_cut)

## Analysis and Inference

The real cepstrum was computed from a normalized speech signal in `test.wav` using log-magnitude spectrum and inverse Fourier transform. The selected frame showed sufficient energy and relatively low zero-crossing rate, which is consistent with a voiced region. This is important because cepstral pitch estimation relies on quasi-periodicity, which is stronger in voiced segments than in unvoiced or silence regions.

In the cepstrum plot of the selected frame, the low-quefrency coefficients represented slowly varying vocal tract characteristics, while a clear peak appeared in the mid-quefrency range corresponding to glottal excitation periodicity. This confirms the source-filter interpretation of speech production: the vocal tract contributes spectral envelope information and the excitation contributes periodic structure.

For pitch estimation, the peak search was constrained to 5 ms to 20 ms, which corresponds to approximately 50 Hz to 200 Hz and lies in a typical voiced speech range. The detected peak gave a direct estimate of pitch period, and inversion provided the fundamental frequency $F_0$. The estimate is stable because the search excludes very low-quefrency components dominated by envelope effects and high-quefrency regions where noise-like fluctuations become dominant.

Cepstral liftering successfully separated coarse and fine spectral components. Low-pass liftering retained only small quefrencies and produced a smooth log-spectral envelope associated with vocal tract resonances. High-pass liftering removed this envelope and emphasized excitation-related fine ripples and harmonic detail. This separation is valuable in speech analysis and coding because envelope cues are strongly related to phonetic content, while excitation cues are strongly related to pitch and voicing.

Overall, the experiment demonstrates that cepstral analysis provides a compact and physically meaningful representation of speech. It supports robust pitch estimation in voiced frames and enables practical source-filter decomposition using simple quefrency-domain filtering. The observed behavior across all steps is consistent with standard speech production theory and validates cepstrum-based processing for speech feature analysis.